In [7]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import fsolve
from CCA_utils import compute_lcl_usd

## M1 CCA Functions: Convenience Yield in Asset Process

Modified Merton with continuous yield $y$ on the sovereign asset:

$$d_1 = \frac{\ln(V/B_f) + (r_f - y + \tfrac{1}{2}\sigma_V^2)\,T}{\sigma_V\sqrt{T}}$$

Call (LCL): $\;V\,e^{-yT}\,N(d_1) - B_f\,e^{-r_f T}\,N(d_2)$  
Vol transfer: $\;\text{LCL}\cdot\sigma_{\text{LCL}} = V\,e^{-yT}\,\sigma_V\,N(d_1)$  
Put: $\;B_f\,e^{-r_f T}\,N(-d_2) - V\,e^{-yT}\,N(-d_1)$

In [8]:
def CCA_system_cy(unknowns, LCL_usd, sigma_lcl, B_f, r_f, gy, T):
    """gy = gamma * y (pre-multiplied)"""
    V, sigma_V = unknowns
    if V <= 0 or sigma_V <= 0:
        return (1e10, 1e10)

    d1 = (np.log(V / B_f) + (r_f - gy + 0.5 * sigma_V**2) * T) / (sigma_V * np.sqrt(T))
    d2 = d1 - sigma_V * np.sqrt(T)

    Ve = V * np.exp(-gy * T)
    eq1 = Ve * norm.cdf(d1) - B_f * np.exp(-r_f * T) * norm.cdf(d2) - LCL_usd
    eq2 = Ve * sigma_V * norm.cdf(d1) - LCL_usd * sigma_lcl
    return (eq1, eq2)


def solve_CCA_cy(LCL_usd, sigma_lcl, B_f, r_f, gy, T):
    if any(np.isnan(x) or x <= 0 for x in [LCL_usd, sigma_lcl, B_f]):
        return {'V': np.nan, 'sigma_V': np.nan, 'converged': False}
    if np.isnan(gy):
        return {'V': np.nan, 'sigma_V': np.nan, 'converged': False}

    V0 = LCL_usd + B_f
    sig0 = sigma_lcl * LCL_usd / V0

    try:
        sol, info, ier, msg = fsolve(
            CCA_system_cy, x0=(V0, sig0),
            args=(LCL_usd, sigma_lcl, B_f, r_f, gy, T),
            full_output=True
        )
        V, sig = sol
        ok = (ier == 1) and (V > 0) and (sig > 0)
        return {'V': V if ok else np.nan, 'sigma_V': sig if ok else np.nan, 'converged': ok}
    except Exception:
        return {'V': np.nan, 'sigma_V': np.nan, 'converged': False}


def compute_risk_cy(V, sigma_V, B_f, r_f, gy, T):
    nans = {'d2': np.nan, 'default_prob': np.nan, 'credit_spread_bps': np.nan,
            'put_value': np.nan, 'risky_debt': np.nan, 'leverage': np.nan}
    if any(np.isnan(x) for x in [V, sigma_V, B_f, gy]) or V <= 0 or sigma_V <= 0:
        return nans

    d1 = (np.log(V / B_f) + (r_f - gy + 0.5 * sigma_V**2) * T) / (sigma_V * np.sqrt(T))
    d2 = d1 - sigma_V * np.sqrt(T)

    Ve = V * np.exp(-gy * T)
    default_prob = norm.cdf(-d2)

    put = B_f * np.exp(-r_f * T) * norm.cdf(-d2) - Ve * norm.cdf(-d1)
    put = max(put, 0.0)

    df_debt = B_f * np.exp(-r_f * T)
    risky_debt = df_debt - put

    if risky_debt > 0:
        y_yield = np.log(B_f / risky_debt) / T
        spread_bps = (y_yield - r_f) * 10000
    else:
        spread_bps = np.nan

    return {
        'd2': d2,
        'default_prob': default_prob,
        'credit_spread_bps': spread_bps,
        'put_value': put,
        'risky_debt': risky_debt,
        'leverage': B_f / V
    }

## Baseline Model Panel

In [9]:
study_sovereigns = [
    'Saudi Arabia', 'Abu Dhabi', 'Dubai', 'Qatar', 'Colombia',
    'Mexico', 'Brazil', 'Egypt', 'Malaysia','Indonesia', 'Philippines', 'Turkey', 'Chile', 'China',
    'South Africa', 'South Korea', 'Thailand']

cca_panel_df = pd.read_csv('../data/processed/CCA_V2/CCA_panel.csv')
cca_panel_df['date'] = pd.to_datetime(cca_panel_df['date'])
cca_panel_df = cca_panel_df[cca_panel_df['country'].isin(study_sovereigns)].copy()

T = 5.0
vol_window = 12
freq = 'ME'
gamma = 0.15

cca_panel_df.set_index(['date','country'], inplace=True)
cca_panel_df = (
    cca_panel_df
    .groupby('country')
    .resample(freq, level='date')
    .last()
)
cca_panel_df.reset_index(inplace=True)

## Merge Futures & Compute Convenience Yield

In [10]:
# Load futures
oil_futures = pd.read_csv('../data/processed/Oil/oil_futures.csv')
oil_futures['date'] = pd.to_datetime(oil_futures['date'], format='%d.%m.%Y')
oil_futures = oil_futures.sort_values('date')


oil_prices = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv').sort_values('date')
oil_prices['date'] = pd.to_datetime(oil_prices['date'], format='%m/%d/%y')
oil_prices = oil_prices.sort_values('date')


# Ensure your main panel is also sorted by date
cca_panel_df = cca_panel_df.reset_index()
cca_panel_df = cca_panel_df.sort_values('date')

# 2. Perform the Directional Merge
# 'direction="nearest"' finds the closest date, whether it is before or after.
cca_panel_df = pd.merge_asof(
    cca_panel_df, 
    oil_prices[['date', 'Brent']], 
    on='date', 
    direction='nearest'
)

cca_panel_df = pd.merge_asof(
    cca_panel_df, 
    oil_futures[['date', 'Brent_12m']], 
    on='date', 
    direction='nearest'
)




# Compute convenience yield per row (using each country's own risk-free rate)
T_fut = 12/12
cca_panel_df['log_basis'] = np.log(cca_panel_df['Brent_12m'] / cca_panel_df['Brent'])
cca_panel_df['conv_yield'] = cca_panel_df['risk_free_rate'] - (1/T_fut) * cca_panel_df['log_basis']

# Restore panel structure
cca_panel_df.set_index(['date', 'country'], inplace=True)
cca_panel_df = (
    cca_panel_df
    .groupby('country')
    .resample(freq, level='date')
    .last()
)
cca_panel_df.reset_index(inplace=True)
cca_panel_df

,country,date,index,cds_spread,fx_rate,domestic_rate,risk_free_rate,monetary_base_bn_local,external_debt_bn_usd,domestic_debt_bn_local,Brent,Brent_12m,log_basis,conv_yield
0,Brazil,2014-01-31,0,205.4300,2.412662,0.13400,0.0352,560.240,482.7710,955.827,107.13,101.63,-0.052704,0.087904
1,Brazil,2014-02-28,1,170.3900,2.338306,0.12700,0.0338,566.324,482.7710,976.705,109.10,104.26,-0.045377,0.079177
2,Brazil,2014-03-31,2,158.0500,2.261778,0.12700,0.0338,566.324,482.7710,976.705,107.03,103.37,-0.034794,0.068594
3,Brazil,2014-04-30,3,148.8900,2.244467,0.12800,0.0335,573.521,482.7710,1014.894,107.78,102.63,-0.048962,0.082462
4,Brazil,2014-05-31,4,139.7800,2.240746,0.12105,0.0312,562.889,482.7710,1043.844,109.56,103.89,-0.053140,0.084340
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1975,Turkey,2024-08-31,1975,265.7500,34.071550,0.26730,0.0425,6157.195,491.9303,393.797,78.89,73.76,-0.067238,0.109738
1976,Turkey,2024-09-30,1976,267.8701,34.129693,0.26730,0.0425,6157.195,491.9303,393.797,71.95,70.81,-0.015971,0.058471
1977,Turkey,2024-10-31,1977,271.2500,34.246575,0.26630,0.0410,6175.970,491.9303,396.645,73.19,71.08,-0.029253,0.070253
1978,Turkey,2024-11-30,1978,251.1200,34.722222,0.27380,0.0463,6067.889,491.9303,392.845,73.20,69.95,-0.045415,0.091715


## Run M1

In [11]:
results = pd.DataFrame()

for country, group in cca_panel_df.groupby('country'):

    df = group.copy().sort_values('date').reset_index(drop=True)

    r_d = df['domestic_rate']
    r_f = df['risk_free_rate']
    M_bn = df['monetary_base_bn_local']
    dom_D_bn = df['domestic_debt_bn_local']
    ext_D_bn = df['external_debt_bn_usd']
    fx_rate = df['fx_rate']

    # LCL unchanged from M0
    df['LCL_usd'] = [
        compute_lcl_usd(m, bd, fx, rd, rf, T)
        for m, bd, fx, rd, rf in zip(
            M_bn, dom_D_bn, fx_rate, r_d, r_f
        )
    ]

    df['B_f'] = ext_D_bn

    ann_factor = np.sqrt(52) if freq == 'W' else np.sqrt(12)
    log_ret = np.log(df['LCL_usd'] / df['LCL_usd'].shift(1))
    df['sigma_lcl'] = log_ret.rolling(window=vol_window).std() * ann_factor

    # Pre-multiply gamma * y
    df['gy'] = gamma * df['conv_yield']

    out = {k: [] for k in ['implied_V', 'implied_sigma_V', 'cca_converged',
                            'distance_to_distress', 'default_prob',
                            'model_spread_bps', 'put_value', 'risky_debt',
                            'leverage']}

    for i, row in df.iterrows():
        gy_i = row['gy'] if pd.notna(row['gy']) else 0.0

        cca = solve_CCA_cy(row['LCL_usd'], row['sigma_lcl'], row['B_f'],
                           r_f.iloc[i], gy_i, T)

        risk = compute_risk_cy(cca['V'], cca['sigma_V'], row['B_f'],
                               r_f.iloc[i], gy_i, T)

        out['implied_V'].append(cca['V'])
        out['implied_sigma_V'].append(cca['sigma_V'])
        out['cca_converged'].append(cca['converged'])
        out['distance_to_distress'].append(risk['d2'])
        out['default_prob'].append(risk['default_prob'])
        out['model_spread_bps'].append(risk['credit_spread_bps'])
        out['put_value'].append(risk['put_value'])
        out['risky_debt'].append(risk['risky_debt'])
        out['leverage'].append(risk['leverage'])

    for col, vals in out.items():
        df[col] = vals

    results = pd.concat([results, df])

START_DATE = '2015-01-01'
END_DATE = '2024-12-31'

results = results[
    (results['date'] >= START_DATE) & (results['date'] <= END_DATE)
].copy()

In [12]:
results.to_csv('../output/results/M1_drift_results_5YCDS.csv')